# Food lists creation

This notebook processes the [FNDDS foods list](https://www.ars.usda.gov/northeast-area/beltsville-md-bhnrc/beltsville-human-nutrition-research-center/food-surveys-research-group/docs/fndds-download-databases/) and create final lists of foods across different categories, to be used in the benchmark construction task (task 2). It utilizes the FNDDS food category codes which can be found in page 39-43 of [this document](https://www.ars.usda.gov/ARSUserFiles/80400530/pdf/fndds/2021_2023_FNDDS_Doc.pdf). 

The data files needed to run this notebook should be placed in a `data` folder under the same directory, which contains these files: 

* `food_list.csv`: a list of 9640 foods found in the FNDDS section of the USDA website. 
* `food_tagging.csv`: detailed nutritional information about these 9640 food items. 

In the same directory, there should also be a `processed_data` folder to store the output files corresponding to the below **7 main categories** of foods: 

* `reduced_mixed_dishes.csv`: 662 food items
* `reduced_meat_seafood.csv`: 192 food items
* `reduced_processed_meat.csv`: 67 food items
* `reduced_plant_protein.csv`: 33 food items
* `reduced_breads.csv`: 152 food items
* `reduced_baked_desserts.csv`: 189 food items
* `reduced_vegetables_potatoes.csv`: 151 food items

Details on the filtering process can be found in each section below. 

## Imports

In [62]:
import pandas as pd
import re
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import os
import time
from autogen import ConversableAgent
from dotenv import load_dotenv
import logging

## Filtering step 1: Applicable to all 7 food categories:

* We first filter for items from sub-categories (`WWEIA_desc`) under a specific food category. 
    
    - For example, mixed dishes can be identified through these WWEIA descriptions, to name a few: 'Meat mixed dishes', 'Poultry mixed dishes', 'Seafood mixed dishes', 'Bean, pea, legume dishes', 'Vegetable dishes', etc. 

* We then remove special characters from the food description and get the first `n` number of foods (usually 2 or 3 depending on the category and on how intense we want the filtering to be, with higher `num_first_words` corresponding to less intense filtering). The purpose of this step is to make sure we can remove near-similar foods that might not necessarily have the exact same name, but they can be considered duplicates. Foods with the same first few words will be grouped together and undergo a selection process. 

    - For example, these foods are almost the same, with the only difference being the type of protein: 58136140 "Lo mein, with pork", 58136130 "Lo mein, with shrimp", 58136150 "Lo mein, with beef", 58136160 "Lo mein, with chicken". To make sure the we have a manageable foods list for latter tasks, we can choose to keep only one food item out of this list of near-similar foods. 

* After grouping foods with similar first few words together, we will select which food to keep (usually one food item remains for most cases, but there might be more than one food item being kept if there is a tie). The general idea is we want to keep foods that have the most nutritional information, that is, the most number of nutrition tags, since they will be helpful in constructing question-answer pairs of varying difficulty levels. The criteria are as follows: 

    - We first look at 16 primary nutrition tags (8 main categories: calorie, protein, carb, sugar, fiber, saturated fat, cholesterol, sodium - each with 2 columns corresponding to high vs. low levels) and get the foods with the highest count in these 16 primary nutrition columns.

    - If there is a tie, we then consider 16 secondary nutrition tags (8 main categories: calcium, phosphorous, potassium, iron, folic acid, vitamin C, vitamin D, vitamin B12 - each with 2 columns corresponding to high vs. low levels) and get the foods with the highest count in these 16 secondary nutrition columns. 

    - If there is still a tie, we randomly select one food item. 

* For many food categories (specifically: mixed dishes, processed meat, baked desserts, breads), no more filtering is needed after this step, since all food items selected at the end of this step can be used in the recommender system already. 

* However, for other categories (meat & seafood, plant protein, vegetables & potatoes), we need to execute another filtering step to make sure we remove foods that cannot be used in a recommender system, such as raw foods, or foods that have not been sufficiently processed or cooked and not yet ready for human consumption.

In [ ]:
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

def process_foods(category, wweia_desc_filters, num_first_words):
    file_path = '../data/foods_list.csv'
    foods_df = pd.read_csv(file_path)
    
    # Filter foods_df for rows where WWEIA_desc is in the specified list
    filtered_foods_df = foods_df[foods_df['WWEIA_desc'].isin(wweia_desc_filters)]
    
    output_file_path = f'../processed_data/{category}.csv'
    filtered_foods_df.to_csv(output_file_path, index=False)
    
    print("Original food list before processing:")
    print(f"Number of rows: {len(filtered_foods_df)}")
    print(f"Number of unique food descriptions: {filtered_foods_df['food_desc'].nunique()}")
    print(f"Number of unique food id's: {filtered_foods_df['food_id'].nunique()}")

    mixed_dishes_df = pd.read_csv(output_file_path)
    food_tagging_df = pd.read_csv('../data/food_tagging.csv')

    # Remove special characters and get the first few words of a food description
    def get_first_words(food_desc, num_words=3):
        clean_desc = re.sub(r'[^a-zA-Z0-9\s]', '', food_desc).lower()
        return ' '.join(clean_desc.split()[:num_words])

    mixed_dishes_df['first_words'] = mixed_dishes_df['food_desc'].apply(lambda x: get_first_words(x, num_words=num_first_words))

    # Group foods by their first few words
    grouped_foods = mixed_dishes_df.groupby('first_words')

    # Count the number of non-zero nutritional tags in specified columns
    def count_non_zero_nutrition_tags(row, columns):
        return row[columns].sum()

    # Primary and secondary sets of nutritional columns
    secondary_nutrition_columns = [
        'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]

    primary_nutrition_columns = [
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium'
    ]

    # Pick one food from each group based on the nutritional tag count
    selected_foods = []

    for group_name, group in grouped_foods:
        group_food_ids = group['food_id']
        
        group_foods_with_tags = pd.merge(
            group, 
            food_tagging_df, 
            on='food_id',
            how='inner'
        )
        
        # Find the food with the most non-zero nutritional tags (primary set)
        group_foods_with_tags['non_zero_primary_count'] = group_foods_with_tags.apply(count_non_zero_nutrition_tags, axis=1, columns=primary_nutrition_columns)
        
        # Get the foods with the highest count in primary nutrition columns
        top_primary_foods = group_foods_with_tags.sort_values(by='non_zero_primary_count', ascending=False)
        top_primary_count = top_primary_foods['non_zero_primary_count'].iloc[0]
        tied_foods = top_primary_foods[top_primary_foods['non_zero_primary_count'] == top_primary_count]
        
        # If there's a tie, compare by the secondary set of nutritional columns
        if len(tied_foods) > 1:
            tied_foods['non_zero_secondary_count'] = tied_foods.apply(count_non_zero_nutrition_tags, axis=1, columns=secondary_nutrition_columns)
            top_secondary_foods = tied_foods.sort_values(by='non_zero_secondary_count', ascending=False)
            top_secondary_count = top_secondary_foods['non_zero_secondary_count'].iloc[0]
            tied_foods_secondary = top_secondary_foods[top_secondary_foods['non_zero_secondary_count'] == top_secondary_count]
            
            # If there's still a tie, randomly choose one
            if len(tied_foods_secondary) > 1:
                selected_food = tied_foods_secondary.sample(1).iloc[0]
            else:
                selected_food = tied_foods_secondary.iloc[0]
        else:
            selected_food = tied_foods.iloc[0]

        selected_foods.append(selected_food)

    selected_foods_df = pd.DataFrame(selected_foods)

    selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_id'])

    selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_desc'])

    # Columns to keep in the final file
    columns_to_keep = [
        "food_id", "food_desc", "WWEIA_desc", "ingredient_desc", "calorie", "protein", "carb", 
        "sugar", "fiber", "saturated_fat", "cholesterol", "sodium", "calcium", "phosphorus", 
        "potassium", "iron", "folic_acid", "vitamin_c", "vitamin_d", "vitamin_b12", 
        "low_calorie", "high_calorie", "low_protein", "high_protein", "low_carb", "high_carb", 
        "low_sugar", "high_sugar", "low_fiber", "high_fiber", "low_saturated_fat", 
        "high_saturated_fat", "low_cholesterol", "high_cholesterol", "low_sodium", "high_sodium", 
        "low_calcium", "high_calcium", "low_phosphorus", "high_phosphorus", "low_potassium", 
        "high_potassium", "low_iron", "high_iron", "low_folic_acid", "high_folic_acid", 
        "low_vitamin_c", "high_vitamin_c", "low_vitamin_d", "high_vitamin_d", 
        "low_vitamin_b12", "high_vitamin_b12"
    ]

    selected_foods_df = selected_foods_df[columns_to_keep]

    print("------------------------------\nFood list after processing: ")
    print(f"Number of rows: {len(selected_foods_df)}")
    print(f"Number of unique food descriptions: {selected_foods_df['food_desc'].nunique()}")
    print(f"Number of unique food id's: {selected_foods_df['food_id'].nunique()}")


    selected_foods_df.to_csv(f'../processed_data/reduced_{category}.csv', index=False)
    
    print(f"------------------------------\nProcessed category '{category}' and saved reduced food list.")


In [55]:
category = "mixed_dishes"
wweia_desc_filters = [
    'Meat mixed dishes',
    'Poultry mixed dishes',
    'Seafood mixed dishes',
    'Bean, pea, legume dishes',
    'Vegetable dishes',
    'Rice mixed dishes',
    'Pasta mixed dishes, excludes macaroni & cheese',
    'Macaroni and cheese',
    'Turnovers and other grain-based items',
    'Fried rice and lo/chow mein',
    'Stir-fry and soy-based sauce mixtures',
    'Egg rolls, dumplings, sushi',
    'Burritos and tacos',
    'Nachos',
    'Other Mexican mixed dishes',
    'Pizza',
    'Burgers',
    'Frankfurter sandwiches',
    'Chicken fillet sandwiches',
    'Egg/breakfast sandwiches',
    'Cheese sandwiches',
    'Peanut butter and jelly sandwiches',
    'Seafood sandwiches',
    'Deli and cured meat sandwiches',
    'Meat and BBQ sandwiches',
    'Vegetable sandwiches/burgers',
    'Soups, broth-based',
    'Soups, cream-based',
    'Ramen and Asian broth-based soups'
]

process_foods(category, wweia_desc_filters, num_first_words=3)

Original food list before processing:
Number of rows: 1597
Number of unique food descriptions: 1443
Number of unique food id's: 1597
------------------------------
Food list after processing: 
Number of rows: 662
Number of unique food descriptions: 662
Number of unique food id's: 662
------------------------------
Processed category 'mixed_dishes' and saved reduced food list.


In [56]:
category = "meat_seafood"
wweia_desc_filters = [
'Beef, excludes ground',
'Ground beef',
'Pork',
'Lamb, goat, game',
'Liver and organ meats',
'Chicken, whole pieces',
'Chicken patties, nuggets and tenders',
'Turkey, duck, other poultry',
'Fish',
'Shellfish',
'Eggs and omelets',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Original food list before processing:
Number of rows: 1176
Number of unique food descriptions: 933
Number of unique food id's: 1176
------------------------------
Food list after processing: 
Number of rows: 312
Number of unique food descriptions: 312
Number of unique food id's: 312
------------------------------
Processed category 'meat_seafood' and saved reduced food list.


In [57]:
category = "processed_meat"
wweia_desc_filters = [
'Cold cuts and cured meats',
'Bacon',
'Frankfurters',
'Sausages',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Original food list before processing:
Number of rows: 175
Number of unique food descriptions: 118
Number of unique food id's: 175
------------------------------
Food list after processing: 
Number of rows: 67
Number of unique food descriptions: 67
Number of unique food id's: 67
------------------------------
Processed category 'processed_meat' and saved reduced food list.


In [58]:
category = "plant_protein"
wweia_desc_filters = [
'Beans, peas, legumes',
'Nuts and seeds',
'Soy and meat-alternative products',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Original food list before processing:
Number of rows: 265
Number of unique food descriptions: 243
Number of unique food id's: 265
------------------------------
Food list after processing: 
Number of rows: 100
Number of unique food descriptions: 100
Number of unique food id's: 100
------------------------------
Processed category 'plant_protein' and saved reduced food list.


In [59]:
category = "breads"
wweia_desc_filters = [
'Yeast breads',
'Rolls and buns',
'Bagels and English muffins',
'Tortillas',
'Biscuits, muffins, quick breads',
'Pancakes, waffles, French toast',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Original food list before processing:
Number of rows: 354
Number of unique food descriptions: 308
Number of unique food id's: 354
------------------------------
Food list after processing: 
Number of rows: 152
Number of unique food descriptions: 152
Number of unique food id's: 152
------------------------------
Processed category 'breads' and saved reduced food list.


In [60]:
category = "baked_desserts"
wweia_desc_filters = [
'Cakes and pies',
'Cookies and brownies',
'Doughnuts, sweet rolls, pastries',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Original food list before processing:
Number of rows: 525
Number of unique food descriptions: 383
Number of unique food id's: 525
------------------------------
Food list after processing: 
Number of rows: 189
Number of unique food descriptions: 189
Number of unique food id's: 189
------------------------------
Processed category 'baked_desserts' and saved reduced food list.


In [61]:
category = "vegetables_potatoes"
wweia_desc_filters = [
'Tomatoes',
'Carrots',
'Other red and orange vegetables',
'Broccoli',
'Spinach',
'Lettuce and lettuce salads',
'Other dark green vegetables',
'String beans',
'Cabbage',
'Onions',
'Corn',
'Other starchy vegetables',
'Other vegetables and combinations',
'Fried vegetables',
'Coleslaw, non-lettuce salads',
'Vegetables on a sandwich',
'White potatoes, baked or boiled',
'French fries and other fried white potatoes',
'Mashed potatoes and white potato mixtures',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Original food list before processing:
Number of rows: 1112
Number of unique food descriptions: 1048
Number of unique food id's: 1112
------------------------------
Food list after processing: 
Number of rows: 233
Number of unique food descriptions: 233
Number of unique food id's: 233
------------------------------
Processed category 'vegetables_potatoes' and saved reduced food list.


## Filtering step 2: Not applicable to all food categories:

* This second round of filtering is applicable only to these 3 food categories: meat & seafood, plant protein, vegetables & potatoes. The main goal is to make sure we filter out food items that are not suitable for human consumption (such as raw foods, or foods that have not been processed or cooked sufficiently) and hence, should not be included in a food recommender system. 

* The other 4 categories: mixed dishes, processed meat, baked desserts, breads - need not go through this step since food items in these categories are guaranteed to be suitable for human consumption. 

* The main method of this step is to utilize LLM agents powered by GPT 3.5 Turbo to help us quickly identify foods that are potentially non-recommendable. We can do this simply by prompting the agents with criteria we want to exclude from our final foods list, specifically: 

    - Raw foods
    - Foods that do not include a specific cooking method in their description

In [ ]:
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')

# Load environment variables
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# LLM configuration
llm_config = {
    "model": "gpt-3.5-turbo",
    "api_key": api_key
}

# Define the food recommendation agent
agent = ConversableAgent(
    name="food_recommendation_agent",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

# Dynamically create a recommendable criteria based on the dataset
def create_recommendable_criteria(instructions):
    return f"""
    The food description must represent a dish that is suitable for recommendation to users. 
    {instructions}
    
    Recommendable foods are those that can be part of a recipe or a full dish.
    """

# Determine if a food is recommendable
def is_food_recommendable(food_desc, recommendable_criteria, retries=3):
    prompt = f"""
    You are a food recommendation agent. Your task is to judge whether a food description should be recommended to users or not.
    Follow the criteria below:
    {recommendable_criteria}

    Food Description: "{food_desc}"
    Should this food be recommended? Answer "Yes" or "No" with a brief explanation.
    """
    
    for attempt in range(retries):
        try:
            response = agent.generate_reply(
                messages=[{"content": prompt, "role": "user"}]
            )
            answer = response.lower().strip()
            
            if "yes" in answer:
                return "recommendable", answer
            elif "no" in answer:
                return "non-recommendable", answer
            else:
                return "non-recommendable", "Unclear response: " + answer

        except Exception as e:
            # Log the error
            logging.debug(f"Attempt {attempt + 1} failed for '{food_desc}' with error: {e}")
            time.sleep(2 * (attempt + 1))

    return "non-recommendable", "Error during evaluation after retries"

def process_food_file(category, instructions):
    # Create recommendable criteria based on the instructions
    recommendable_criteria = create_recommendable_criteria(instructions)
    
    # Read the input file
    file_path = f'../processed_data/reduced_{category}.csv'
    foods_df = pd.read_csv(file_path)

    # Determine if each food is recommendable
    food_results = []
    for idx, row in foods_df.iterrows():
        food_desc = row['food_desc']
        recommendable_flag, reasoning = is_food_recommendable(food_desc, recommendable_criteria)
        
        food_results.append({
            "food_id": row['food_id'],
            "food_desc": food_desc,
            "WWEIA_desc": row['WWEIA_desc'],
            "ingredient_desc": row.get('ingredient_desc', ''),
            "recommendable_flag": recommendable_flag,
            "reasoning": reasoning
        })
        time.sleep(2)

    food_results_df = pd.DataFrame(food_results)
    
    # Save initial output with recommendations
    intermediate_output_path = f'../processed_data/reduced_{category}_with_recommendations.csv'
    food_results_df.to_csv(intermediate_output_path, index=False)
    print(f"Intermediate results saved to {intermediate_output_path}")

    # Filter rows where recommendable_flag is "recommendable"
    recommendable_df = food_results_df[food_results_df['recommendable_flag'] == "recommendable"]

    # Keep only necessary columns
    recommendable_df = recommendable_df[["food_id", "food_desc", "WWEIA_desc", "ingredient_desc"]]

    # Load the food_tagging data to get nutrition tags
    food_tagging_df = pd.read_csv('../data/food_tagging.csv')

    # Columns to join from food_tagging.csv
    food_tagging_columns = [
        "calorie", "protein", "carb", "sugar", "fiber", "saturated_fat", "cholesterol",
        "sodium", "calcium", "phosphorus", "potassium", "iron", "folic_acid", "vitamin_c",
        "vitamin_d", "vitamin_b12", "low_calorie", "high_calorie", "low_protein", "high_protein",
        "low_carb", "high_carb", "low_sugar", "high_sugar", "low_fiber", "high_fiber",
        "low_saturated_fat", "high_saturated_fat", "low_cholesterol", "high_cholesterol",
        "low_sodium", "high_sodium", "low_calcium", "high_calcium", "low_phosphorus",
        "high_phosphorus", "low_potassium", "high_potassium", "low_iron", "high_iron",
        "low_folic_acid", "high_folic_acid", "low_vitamin_c", "high_vitamin_c", "low_vitamin_d",
        "high_vitamin_d", "low_vitamin_b12", "high_vitamin_b12"
    ]

    # Left join on food_id
    final_df = recommendable_df.merge(food_tagging_df[["food_id"] + food_tagging_columns], on="food_id", how="left")

    print("------------------------------\nList of recommendable foods: ")
    print(f"Number of rows: {len(final_df)}")
    print(f"Number of unique food descriptions: {final_df['food_desc'].nunique()}")
    print(f"Number of unique food id's: {final_df['food_id'].nunique()}")

    output_path = f'../processed_data/reduced_{category}.csv'
    final_df.to_csv(output_path, index=False)
    print(f"------------------------------\nFinal results saved to {output_path}")


In [33]:
category = "meat_seafood"
instructions = """
Avoid foods that: 
1. Are raw
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

Intermediate results saved to ../processed_data/reduced_meat_seafood_with_recommendations.csv
------------------------------
List of recommendable foods: 
Number of rows: 192
Number of unique food descriptions: 192
Number of unique food id's: 192
------------------------------
Final results saved to ../processed_data/reduced_meat_seafood.csv


In [38]:
category = "plant_protein"
instructions = """
Avoid foods that: 
1. Are raw or unprocessed
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

Intermediate results saved to ../processed_data/reduced_plant_protein_with_recommendations.csv
------------------------------
List of recommendable foods: 
Number of rows: 33
Number of unique food descriptions: 33
Number of unique food id's: 33
------------------------------
Final results saved to ../processed_data/reduced_plant_protein.csv


In [54]:
category = "vegetables_potatoes"
instructions = """
Avoid foods that: 
1. Are raw vegetables/potatoes or uncooked vegetables/potatoes
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

Intermediate results saved to ../processed_data/reduced_vegetables_potatoes_with_recommendations.csv
------------------------------
List of recommendable foods: 
Number of rows: 151
Number of unique food descriptions: 151
Number of unique food id's: 151
------------------------------
Final results saved to ../processed_data/reduced_vegetables_potatoes.csv
